# Stadium Intro Analysis

Parameterized notebook for comparing individual stadium weather and play statistics against league averages.

**To switch teams**: Change `TEAM` in the cell below and re-run all cells. Season range auto-loads from `team_parameters.csv` but can be overridden.

**To run all teams**: Set `RUN_ALL_TEAMS = True` to batch-generate graphics for all 30 stadiums. Plots are saved to `intro_graphics/{TEAM}/` and not displayed inline to save memory.

In [ ]:
# ============================================================
# PARAMETERS — change these to switch teams
# ============================================================
TEAM = 'BOS'           # <- change this (see team_parameters.csv for all codes)
RUN_ALL_TEAMS = False  # <- set True to batch-generate all 30 teams

# Season range auto-loaded from team_parameters.csv.
# Uncomment below to override:
# SEASON_START = 2021
# SEASON_END = 2025

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import os
import warnings
warnings.filterwarnings('ignore')

import statsmodels.api as sm
from matplotlib.colors import Normalize, TwoSlopeNorm

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# --- Load team parameters ---
params_df = pd.read_csv('team_parameters.csv')

DATASETS_DIR = os.path.join('..', 'data')

# --- Load or generate league weather data (2021-2025, all stadiums) ---
league_file = os.path.join(DATASETS_DIR, 'league_weather_2021_2025.csv')

if os.path.exists(league_file):
    league_data = pd.read_csv(league_file)
    league_data['game_date'] = pd.to_datetime(league_data['game_date'])
    print(f"League data loaded: {len(league_data)} games")
else:
    print("league_weather_2021_2025.csv not found — generating from team CSVs...")
    weather_cols = ['game_pk', 'game_date', 'season', 'temp_f', 'wspd_mph', 'rhum', 'pres',
                    'wdir', 'wind_cf', 'wind_lcf', 'wind_rcf']
    all_frames = []
    for _, row in params_df.iterrows():
        team_file = os.path.join(DATASETS_DIR, row['dataset_file'])
        if not os.path.exists(team_file):
            continue
        df = pd.read_csv(team_file)
        df = df[(df['season'] >= 2021) & (df['season'] <= 2025)]
        available = [c for c in weather_cols if c in df.columns]
        df = df[available].copy()
        df['home_team'] = row['team_code']
        all_frames.append(df)

    league_data = pd.concat(all_frames, ignore_index=True)
    league_data['game_date'] = pd.to_datetime(league_data['game_date'])
    league_data.to_csv(league_file, index=False)
    print(f"Generated league_weather_2021_2025.csv: {len(league_data)} games from {len(all_frames)} teams")

# --- Determine which teams to process ---
if RUN_ALL_TEAMS:
    teams_to_process = params_df['team_code'].tolist()
else:
    teams_to_process = [TEAM]

print(f"Teams to process: {teams_to_process}")

In [ ]:
# ============================================================
# REGRESSION CONFIGURATION
# ============================================================
DEPENDENT_VARS = {
    'away_runs_scored': 'Away Runs Scored',
    'away_bat_k':       'Away Strikeouts',
    'away_bat_hr_h_ratio': 'Away HR:H Ratio',
}

INDEPENDENT_VARS = [
    'temp_f', 'rhum', 'pres', 'prcp', 'wspd_mph',
    'wind_cf', 'wind_lcf', 'wind_rcf',
]

IV_DISPLAY_NAMES = {
    'temp_f': 'Temp (F)',
    'rhum': 'Humidity (%)',
    'pres': 'Pressure (hPa)',
    'prcp': 'Precip (mm)',
    'wspd_mph': 'Wind Speed (mph)',
    'wind_cf': 'Wind to CF',
    'wind_lcf': 'Wind to LCF',
    'wind_rcf': 'Wind to RCF',
    'const': 'Intercept',
}

SUBSETS = {
    'all': 'All Games',
    'day': 'Day Games (before 5 PM)',
    'night': 'Night Games (5 PM+)',
}

# ============================================================
# HEATMAP CONFIGURATION
# ============================================================
WEATHER_VARS = {
    'temp_f':   {'label': 'Temperature',  'unit': '\u00b0F',  'n_bins': 5, 'fmt': '.0f'},
    'wspd_mph': {'label': 'Wind Speed',   'unit': ' mph',     'n_bins': 5, 'fmt': '.0f'},
    'rhum':     {'label': 'Humidity',      'unit': '%',        'n_bins': 5, 'fmt': '.0f'},
    'pres':     {'label': 'Pressure',      'unit': ' hPa',     'n_bins': 5, 'fmt': '.0f'},
}

BASEBALL_STATS = {
    'away_runs_scored':    {'label': 'Away Runs',       'fmt': '.1f'},
    'away_bat_hr':         {'label': 'Away Home Runs',  'fmt': '.2f'},
    'away_bat_k':          {'label': 'Away Strikeouts', 'fmt': '.1f'},
    'away_bat_bb':         {'label': 'Away Walks',      'fmt': '.1f'},
    'away_bat_hr_h_ratio': {'label': 'Away HR:H Ratio', 'fmt': '.3f'},
}

# ============================================================
# WIND PROJECTION CONFIGURATION
# ============================================================
WIND_PROJ_VARS = {
    'wind_cf':  {'label': 'Wind to CF',  'color': '#c0392b'},
    'wind_lcf': {'label': 'Wind to LCF', 'color': '#2980b9'},
    'wind_rcf': {'label': 'Wind to RCF', 'color': '#27ae60'},
}
MIN_GAMES_PER_BIN = 5
N_WIND_BINS = 10

print("Configuration loaded.")

## Helper Functions

All plotting, regression, and heatmap functions used by `run_team_analysis()`.

In [ ]:
# ============================================================
# WEATHER DISTRIBUTION PLOTS
# ============================================================

def plot_weather_distribution(team_window, league_data, col, xlabel, unit,
                              stadium_name, graphics_dir, show=True):
    """Plot team vs league distribution for a weather variable."""
    fig, ax = plt.subplots(figsize=(10, 6))

    team_vals = team_window[col].dropna()
    league_vals = league_data[col].dropna()

    if col == 'rhum':
        bins = np.linspace(0, 100, 35)
    elif col == 'wspd_mph':
        bins = np.linspace(0, max(team_vals.max(), league_vals.max()) + 1, 35)
    else:
        bins = np.linspace(
            min(team_vals.min(), league_vals.min()) - 2,
            max(team_vals.max(), league_vals.max()) + 2,
            40 if col == 'temp_f' else 35
        )

    ax.hist(league_vals, bins=bins, density=True, alpha=0.5, color='steelblue',
            edgecolor='white', linewidth=0.5,
            label=f'League Avg (mean: {league_vals.mean():.1f}{unit})')
    ax.hist(team_vals, bins=bins, density=True, alpha=0.6, color='firebrick',
            edgecolor='white', linewidth=0.5,
            label=f'{stadium_name} (mean: {team_vals.mean():.1f}{unit})')

    ax.axvline(league_vals.mean(), color='steelblue', linestyle='--', linewidth=1.5)
    ax.axvline(team_vals.mean(), color='firebrick', linestyle='--', linewidth=1.5)

    ax.set_xlabel(f'{xlabel} ({unit.strip()})', fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    ax.set_title(f'{xlabel} Distribution: {stadium_name} vs League Average (2021-2025)', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()

    filename = f'{col}_distribution.png'
    fig.savefig(os.path.join(graphics_dir, filename), dpi=150, bbox_inches='tight')
    if show:
        plt.show()
    else:
        plt.close(fig)
    return filename


# ============================================================
# OLS REGRESSION FUNCTIONS
# ============================================================

def run_ols(df, dep_var, indep_vars):
    """Run OLS regression with mean-centered independent variables.

    Mean-centering the IVs makes the intercept interpretable:
    it equals the expected value of the DV when all weather
    predictors are at their mean. Slopes are unchanged.
    """
    cols_needed = [dep_var] + indep_vars
    reg_df = df[cols_needed].dropna()
    n_dropped = len(df) - len(reg_df)

    reg_df = reg_df.copy()

    # Mean-center independent variables
    iv_means = {}
    for col in indep_vars:
        iv_means[col] = reg_df[col].mean()
        reg_df[col] = reg_df[col] - iv_means[col]

    X = sm.add_constant(reg_df[indep_vars])
    y = reg_df[dep_var]

    model = sm.OLS(y, X).fit()
    return model, n_dropped, iv_means


def results_to_dataframe(model):
    """Extract a clean summary DataFrame from an OLS result."""
    summary_df = pd.DataFrame({
        'Variable': [IV_DISPLAY_NAMES.get(v, v) for v in model.params.index],
        'Coefficient': model.params.values,
        'Std Error': model.bse.values,
        't-stat': model.tvalues.values,
        'P-value': model.pvalues.values,
    })

    def sig_stars(p):
        if p < 0.001: return '***'
        if p < 0.01:  return '**'
        if p < 0.05:  return '*'
        if p < 0.10:  return '.'
        return ''

    summary_df['Sig'] = summary_df['P-value'].apply(sig_stars)
    summary_df['Coefficient'] = summary_df['Coefficient'].round(4)
    summary_df['Std Error'] = summary_df['Std Error'].round(4)
    summary_df['t-stat'] = summary_df['t-stat'].round(3)
    summary_df['P-value'] = summary_df['P-value'].round(4)
    return summary_df


def render_regression_table(results_df, model, dep_label, subset_label,
                            n_obs, n_dropped, iv_means, stadium_name,
                            season_start, season_end, filepath, show=True):
    """Render regression results as a matplotlib table and save."""
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.axis('off')

    title = (f'OLS Regression: {dep_label} (IVs mean-centered)\n'
             f'{stadium_name} \u2014 {subset_label} ({season_start}-{season_end})')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=20, loc='left')

    table = ax.table(
        cellText=results_df.values.tolist(),
        colLabels=results_df.columns.tolist(),
        cellLoc='center',
        loc='center',
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.0, 1.4)

    for j in range(len(results_df.columns)):
        table[0, j].set_facecolor('#2c3e50')
        table[0, j].set_text_props(color='white', fontweight='bold')

    for i in range(1, len(results_df) + 1):
        for j in range(len(results_df.columns)):
            if i % 2 == 0:
                table[i, j].set_facecolor('#f0f0f0')

    # Intercept value = expected DV at mean weather conditions
    intercept_val = model.params.get('const', np.nan)
    summary_text = (
        f'n = {n_obs}    '
        f'R\u00b2 = {model.rsquared:.4f}    '
        f'Adj R\u00b2 = {model.rsquared_adj:.4f}    '
        f'F = {model.fvalue:.2f} (p = {model.f_pvalue:.4f})    '
        f'Intercept = expected {dep_label} at mean weather ({intercept_val:.2f})'
    )
    if n_dropped > 0:
        summary_text += f'    [{n_dropped} obs dropped]'

    fig.text(0.05, 0.02, summary_text, fontsize=9, fontstyle='italic', color='#555555')
    fig.text(0.95, 0.02, 'Sig: *** p<0.001  ** p<0.01  * p<0.05  . p<0.10',
             fontsize=8, ha='right', color='#888888')

    plt.tight_layout()
    fig.savefig(filepath, dpi=150, bbox_inches='tight', facecolor='white')
    if show:
        plt.show()
    else:
        plt.close(fig)


# ============================================================
# HEATMAP FUNCTIONS
# ============================================================

def bucket_weather_var(df, col, config):
    """Bucket a numeric weather column into quantile bins."""
    series = df[col].dropna()
    buckets = pd.qcut(series, q=config['n_bins'], duplicates='drop')

    label_map = {}
    for interval in buckets.cat.categories:
        left = format(interval.left, config['fmt'])
        right = format(interval.right, config['fmt'])
        label_map[interval] = f"{left}-{right}{config['unit']}"

    labeled = buckets.map(label_map)
    bucket_order = [label_map[iv] for iv in buckets.cat.categories]
    counts = labeled.value_counts().reindex(bucket_order).to_dict()
    return labeled, bucket_order, counts


def render_heatmap(df, weather_col, weather_config, version, stadium_name,
                   season_start, season_end, filepath, show=True):
    """Render a weather-impact heatmap (raw or diff) and save."""
    labeled, bucket_order, counts = bucket_weather_var(df, weather_col, weather_config)
    n_cols = len(bucket_order)
    n_rows = len(BASEBALL_STATS)

    stat_keys = list(BASEBALL_STATS.keys())
    stat_labels = [BASEBALL_STATS[k]['label'] for k in stat_keys]
    stat_fmts = [BASEBALL_STATS[k]['fmt'] for k in stat_keys]

    df_valid = df.loc[labeled.index].copy()
    df_valid['_bucket'] = labeled.values

    matrix = np.full((n_rows, n_cols), np.nan)
    for i, stat in enumerate(stat_keys):
        overall_mean = df[stat].mean()
        grouped = df_valid.groupby('_bucket')[stat].mean()
        for j, bucket_label in enumerate(bucket_order):
            if bucket_label in grouped.index:
                val = grouped[bucket_label]
                matrix[i, j] = val if version == 'raw' else val - overall_mean

    cmap = plt.cm.YlOrRd if version == 'raw' else plt.cm.RdBu_r

    fig_width = max(8, n_cols * 1.8)
    fig, ax = plt.subplots(figsize=(fig_width, 5))
    ax.set_xlim(0, n_cols)
    ax.set_ylim(0, n_rows)
    ax.invert_yaxis()
    ax.set_aspect('equal')
    ax.axis('off')

    for i in range(n_rows):
        row_vals = matrix[i, :]
        valid = row_vals[~np.isnan(row_vals)]
        if len(valid) == 0:
            continue
        vmin, vmax = valid.min(), valid.max()

        if vmin == vmax:
            norm = Normalize(vmin=vmin - 1, vmax=vmax + 1)
        elif version == 'diff':
            abs_max = max(abs(vmin), abs(vmax))
            norm = TwoSlopeNorm(vcenter=0, vmin=-abs_max, vmax=abs_max)
        else:
            norm = Normalize(vmin=vmin, vmax=vmax)

        for j in range(n_cols):
            val = matrix[i, j]
            color = (0.9, 0.9, 0.9, 1.0) if np.isnan(val) else cmap(norm(val))
            rect = plt.Rectangle((j, i), 1, 1, facecolor=color, edgecolor='white', linewidth=1.5)
            ax.add_patch(rect)

            if not np.isnan(val):
                lum = 0.299 * color[0] + 0.587 * color[1] + 0.114 * color[2]
                txt_color = 'white' if lum < 0.5 else 'black'
                prefix = '+' if version == 'diff' and val > 0 else ''
                ax.text(j + 0.5, i + 0.5, f"{prefix}{val:{stat_fmts[i]}}",
                        ha='center', va='center', fontsize=10, fontweight='bold',
                        color=txt_color)

    for j, bucket_label in enumerate(bucket_order):
        n = counts.get(bucket_label, 0)
        ax.text(j + 0.5, -0.15, f"{bucket_label}\n(n={n})",
                ha='center', va='bottom', fontsize=9)

    for i, label in enumerate(stat_labels):
        ax.text(-0.1, i + 0.5, label, ha='right', va='center', fontsize=11)

    version_label = 'Mean' if version == 'raw' else 'Diff from Mean'
    ax.set_title(f'{version_label}: Away Stats by {weather_config["label"]}\n'
                 f'{stadium_name} ({season_start}-{season_end})',
                 fontsize=13, fontweight='bold', pad=30, loc='left')

    plt.tight_layout()
    fig.savefig(filepath, dpi=150, bbox_inches='tight', facecolor='white')
    if show:
        plt.show()
    else:
        plt.close(fig)


# ============================================================
# WIND PROJECTION LINE GRAPH FUNCTION
# ============================================================

def render_wind_line_graph(df, stat_col, stat_config, shared_bins,
                           stadium_name, season_start, season_end,
                           filepath, show=True):
    """Render a line graph showing stat vs wind projection for CF/LCF/RCF."""
    fig, ax = plt.subplots(figsize=(10, 6))

    for wind_col, wind_config in WIND_PROJ_VARS.items():
        series = df[[wind_col, stat_col]].dropna()
        series['_bin'] = pd.cut(series[wind_col], bins=shared_bins, include_lowest=True)

        grouped = series.groupby('_bin')[stat_col].agg(['mean', 'count'])
        grouped = grouped[grouped['count'] >= MIN_GAMES_PER_BIN]

        if len(grouped) == 0:
            continue

        midpoints = [(iv.left + iv.right) / 2 for iv in grouped.index]
        ax.plot(midpoints, grouped['mean'].values, marker='o', markersize=5,
                linewidth=2, color=wind_config['color'], label=wind_config['label'])

    ax.axvline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_xlabel('Wind Projection (mph) \u2014 negative = blowing in, positive = blowing out',
                  fontsize=12)
    ax.set_ylabel(stat_config['label'], fontsize=12)
    ax.set_title(f'{stat_config["label"]} by Wind Projection\n'
                 f'{stadium_name} ({season_start}-{season_end})', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    fig.savefig(filepath, dpi=150, bbox_inches='tight', facecolor='white')
    if show:
        plt.show()
    else:
        plt.close(fig)


print("All helper functions defined.")

## Main Analysis Function

`run_team_analysis()` runs the full pipeline for a single team: loads data, generates weather distribution plots, runs OLS regressions, creates heatmaps, and generates wind projection line graphs.

In [ ]:
def run_team_analysis(team_code, params_df, league_data, show=True):
    """
    Run the full intro analysis for a single team.
    
    Parameters
    ----------
    team_code : str
        Team code (e.g. 'SF', 'BOS')
    params_df : DataFrame
        Team parameters
    league_data : DataFrame
        League-wide weather data for comparison
    show : bool
        If True, display plots inline. If False, save only (for batch mode).
    
    Returns
    -------
    dict with 'team', 'status', 'n_games', 'n_graphics' keys
    """
    # --- Team config ---
    team_row = params_df[params_df['team_code'] == team_code]
    if len(team_row) == 0:
        print(f"  ERROR: Team '{team_code}' not found in team_parameters.csv")
        return {'team': team_code, 'status': 'NOT FOUND', 'n_games': 0, 'n_graphics': 0}
    team_row = team_row.iloc[0]

    team_name = team_row['team_name']
    stadium_name = team_row['stadium_name']
    dataset_file = team_row['dataset_file']
    season_start = int(team_row['data_start_year'])
    season_end = int(team_row['data_end_year'])

    # Override season range if user set it
    if 'SEASON_START' in dir():
        season_start = SEASON_START
    if 'SEASON_END' in dir():
        season_end = SEASON_END

    graphics_dir = os.path.join('intro_graphics', team_code)
    os.makedirs(graphics_dir, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"  {stadium_name} ({team_name}) — {team_code}")
    print(f"  Seasons: {season_start}-{season_end}")
    print(f"  Dataset: {dataset_file}")
    print(f"  Graphics: {os.path.abspath(graphics_dir)}")
    print(f"{'='*60}")

    # --- Load team data ---
    dataset_path = os.path.join(DATASETS_DIR, dataset_file)
    if not os.path.exists(dataset_path):
        print(f"  WARNING: Dataset not found: {dataset_path}")
        return {'team': team_code, 'status': 'NO DATA', 'n_games': 0, 'n_graphics': 0}

    team_data = pd.read_csv(dataset_path)
    team_data['game_date'] = pd.to_datetime(team_data['game_date'])
    team_data = team_data[
        (team_data['season'] >= season_start) &
        (team_data['season'] <= season_end)
    ].copy()

    print(f"  Games: {len(team_data)} ({season_start}-{season_end})")

    n_graphics = 0

    # ==========================================
    # 1. WEATHER DISTRIBUTION PLOTS (4 plots)
    # ==========================================
    team_league_window = team_data[
        (team_data['season'] >= 2021) & (team_data['season'] <= 2025)
    ].copy()

    weather_plots = [
        ('temp_f',   'Temperature',       '\u00b0F'),
        ('wspd_mph', 'Wind Speed',        ' mph'),
        ('rhum',     'Relative Humidity',  '%'),
        ('pres',     'Surface Pressure',   ' hPa'),
    ]

    for col, xlabel, unit in weather_plots:
        plot_weather_distribution(
            team_league_window, league_data, col, xlabel, unit,
            stadium_name, graphics_dir, show=show
        )
        n_graphics += 1

    print(f"  Weather distribution plots: 4 saved")

    # ==========================================
    # 2. OLS REGRESSIONS (9 tables)
    # ==========================================
    team_data['is_day'] = team_data['start_hour'] < 17
    subsets = {
        'all': team_data,
        'day': team_data[team_data['is_day']],
        'night': team_data[~team_data['is_day']],
    }

    for dep_var, dep_label in DEPENDENT_VARS.items():
        for subset_key, subset_label in SUBSETS.items():
            subset_df = subsets[subset_key]
            model, n_dropped, iv_means = run_ols(subset_df, dep_var, INDEPENDENT_VARS)
            results_df = results_to_dataframe(model)

            filename = f'ols_{dep_var}_{subset_key}.png'
            filepath = os.path.join(graphics_dir, filename)

            render_regression_table(
                results_df, model, dep_label, subset_label,
                int(model.nobs), n_dropped, iv_means,
                stadium_name, season_start, season_end,
                filepath, show=show
            )
            n_graphics += 1

    print(f"  OLS regression tables: 9 saved")

    # ==========================================
    # 3. HEATMAPS (8 plots: 4 weather vars x 2 versions)
    # ==========================================
    for weather_col, weather_config in WEATHER_VARS.items():
        for version in ['raw', 'diff']:
            filename = f'heatmap_{weather_col}_{version}.png'
            filepath = os.path.join(graphics_dir, filename)

            render_heatmap(
                team_data, weather_col, weather_config, version,
                stadium_name, season_start, season_end,
                filepath, show=show
            )
            n_graphics += 1

    print(f"  Heatmaps: 8 saved")

    # ==========================================
    # 4. WIND PROJECTION LINE GRAPHS (5 plots)
    # ==========================================
    all_wind_vals = pd.concat([team_data[c].dropna() for c in WIND_PROJ_VARS])
    shared_bins = np.linspace(all_wind_vals.min(), all_wind_vals.max(), N_WIND_BINS + 1)

    for stat_col, stat_config in BASEBALL_STATS.items():
        filename = f'wind_projection_{stat_col}.png'
        filepath = os.path.join(graphics_dir, filename)

        render_wind_line_graph(
            team_data, stat_col, stat_config, shared_bins,
            stadium_name, season_start, season_end,
            filepath, show=show
        )
        n_graphics += 1

    print(f"  Wind projection graphs: 5 saved")
    print(f"  Total graphics: {n_graphics}")

    return {
        'team': team_code,
        'status': 'OK',
        'n_games': len(team_data),
        'n_graphics': n_graphics,
        'stadium': stadium_name,
    }


print("run_team_analysis() defined.")

## Run Analysis

Processes each team in `teams_to_process`. When `RUN_ALL_TEAMS = True`, plots are saved to disk only (not displayed inline) to avoid memory issues with 780+ figures.

In [ ]:
# --- Run analysis for all selected teams ---
show_plots = not RUN_ALL_TEAMS  # Only show inline for single-team mode

if RUN_ALL_TEAMS:
    plt.ioff()  # Suppress inline display for batch mode
    print(f"Batch mode: generating graphics for {len(teams_to_process)} teams (plots saved to disk only)\n")

results_summary = []

for i, team_code in enumerate(teams_to_process):
    if RUN_ALL_TEAMS:
        print(f"\n[{i+1}/{len(teams_to_process)}] Processing {team_code}...")

    try:
        result = run_team_analysis(team_code, params_df, league_data, show=show_plots)
        results_summary.append(result)
    except Exception as e:
        print(f"  ERROR processing {team_code}: {e}")
        results_summary.append({
            'team': team_code, 'status': f'ERROR: {e}',
            'n_games': 0, 'n_graphics': 0
        })

if RUN_ALL_TEAMS:
    plt.ion()  # Re-enable inline display

# --- Summary ---
print(f"\n{'='*60}")
print(f"SUMMARY")
print(f"{'='*60}")
summary_df = pd.DataFrame(results_summary)
print(summary_df.to_string(index=False))

n_ok = sum(1 for r in results_summary if r['status'] == 'OK')
n_total = len(results_summary)
total_graphics = sum(r['n_graphics'] for r in results_summary)
print(f"\n{n_ok}/{n_total} teams completed, {total_graphics} total graphics generated")